In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd drive/Othercomputers/My\ MacBook\ Pro/GPU-Computing/

/content/drive/Othercomputers/My MacBook Pro/GPU-Computing


In [3]:
!pwd

/content/drive/Othercomputers/My MacBook Pro/GPU-Computing


In [4]:
!which nvcc

/usr/local/cuda/bin/nvcc


In [5]:
!g++ -std=c++17 ./serial_image_blur.cpp -o serial-blur && ./serial-blur

elapsed time CPU 45.5409


In [6]:
!nvcc -arch=sm_75 ./parallel_matrix_mul_tiling_opt.cu -o  mm_tiling_opt

In [7]:
!pwd

/content/drive/Othercomputers/My MacBook Pro/GPU-Computing


In [8]:
!ls 


apple.jpg				    README.md
main					    RGB_to_gray_parallel.cu
matrix_multi_tiling_opt.prof.ncu-rep	    RGB_to_gray_serial.cpp
mm_tiling_opt				    run.ipynb
parallel_blur.png			    serial-blur
parallel_gray.png			    serial_blur.png
parallel_image_blur.cu			    serial_gray.png
parallel_matrix_mul.cu			    serial_image_blur.cpp
parallel_matrix_mul_tiling_benchmarking.cu  stb_image.h
parallel_matrix_mul_tiling.cu		    stb_image_write.h
parallel_matrix_mul_tiling_opt.cu	    timeline.prof
parallel-patterns			    vec_profile
profiling.cu				    vector_addition.cu


In [9]:
!nvprof -f -o timeline.prof ./vec_profile

======== Error: Application returned non-zero code -1


In [10]:
!nvprof -m all ./vec_profile

======== Warning: Skipping profiling on device 0 since profiling is not supported on devices with compute capability 7.5 and higher.
                  Use NVIDIA Nsight Compute for GPU profiling and NVIDIA Nsight Systems for GPU tracing and CPU sampling.
                  Refer https://developer.nvidia.com/tools-overview for more details.

======== Error: Application returned non-zero code -1


## NVPROF is deprecated

======== Warning: Skipping profiling on device 0 since profiling is not supported on devices with compute capability 7.5 and higher.
                  Use NVIDIA Nsight Compute for GPU profiling and NVIDIA Nsight Systems for GPU tracing and CPU sampling.
                  Refer https://developer.nvidia.com/tools-overview for more details.

We might need NVIDIA Nsight (new profiling) for new devices

In [11]:
!ncu --set full ./mm_tiling_opt

elapsed time CPU 6443.21
==PROF== Connected to process 8590 (/content/drive/Othercomputers/My MacBook Pro/GPU-Computing/mm_tiling_opt)
==PROF== Profiling "mat_mul_tile_kernel_opt" - 0: 0%....50%....100% - 31 passes
elapsed time GPU 7990.89
Max_Diff 9.15527e-05
==PROF== Disconnected from process 8590
[8590] mm_tiling_opt@127.0.0.1
  mat_mul_tile_kernel_opt(float *, float *, float *, int) (8, 32, 1)x(32, 32, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       585.00
    Elapsed Cycles                cycle    2,380,039
    Memory Throughput                 %        80.54
    DRAM Throughput                   %         4.10
    Duration                         ms         4.07
    L1/TEX Cache Throughput 

In [12]:
!ncu --version


NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2025 NVIDIA Corporation
Version 2025.1.1.0 (build 35528883) (public-release)


##Parallel Patterns

## Convolution


In [26]:
!nvcc -arch=sm_75  -I ./stb ./parallel-patterns/convolution.cu -o conv

./stb/stb_image.h(4276): warning #550-D: variable "old_limit" was set but never used
     unsigned int cur, limit, old_limit;
                              ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

./stb/stb_image.h(5185): warning #550-D: variable "idata_limit_old" was set but never used
                 stbi__uint32 idata_limit_old = idata_limit;
                              ^

./stb/stb_image.h(6972): warning #550-D: variable "out_size" was set but never used
        int out_size = 0;
            ^

./stb/stb_image.h(6973): warning #550-D: variable "delays_size" was set but never used
        int delays_size = 0;
            ^



In [28]:
!./conv

In [29]:
!ncu --set full ./conv

==PROF== Connected to process 12725 (/content/drive/Othercomputers/My MacBook Pro/GPU-Computing/conv)
==PROF== Profiling "convolution_kernel" - 0: 0%....50%....100% - 31 passes
==PROF== Disconnected from process 12725
[12725] conv@127.0.0.1
  convolution_kernel(unsigned char *, unsigned char *, unsigned int, unsigned int) (32, 32, 1)x(32, 32, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       584.98
    Elapsed Cycles                cycle      440,400
    Memory Throughput                 %        29.05
    DRAM Throughput                   %         5.12
    Duration                         us       752.83
    L1/TEX Cache Throughput           %        31.29
    L2 Cache Throughput               % 